# Part 1.1 (extension) — Variable-level description diff for the 5,674 shared `var_id`s

**Scope:** Part 1.4 established that 5,674 `var_id`s are present under the same
name in both the 2024H2 and 2026 variable dictionaries. Part 1.1 checked
content stability at the *table* level (treating each table's variable
descriptions as an unordered set and computing Jaccard similarity between
vintages). Neither checked whether an individual shared `var_id`'s own
description text is the same string in both dictionaries. This notebook
does that: a direct, one-row-per-`var_id` text comparison across the full
5,674-variable shared set, not a sample.

**Data sources:** `data/CA_2024H2/VARIABLE_MAPPING/*.parquet` (2024H2 dictionary)
and `data/variable_mapping_2026.csv` (2026 dictionary) — both already cached
locally, no new GCS pull needed. `data/part1_4_diff_results.pkl` (Part 1.4's
cached `shared_vars` set, 5,674 entries) and `data/part1_1_table_self_similarity.csv`
(Part 1.1's per-`table_id` Jaccard scores) are reused directly rather than
recomputed, per this project's working convention of not re-deriving cached
results.


In [1]:
import os
os.chdir("/Users/wiame.ichane/arima_v24_v26")

import re
import glob
import pickle
import difflib
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 80)

vm2024 = pd.concat([pd.read_parquet(p) for p in glob.glob("data/CA_2024H2/VARIABLE_MAPPING/*.parquet")], ignore_index=True)
vm2024.columns = [c.lower() for c in vm2024.columns]
vm2026 = pd.read_csv("data/variable_mapping_2026.csv")

with open("data/part1_4_diff_results.pkl", "rb") as f:
    part1_4 = pickle.load(f)
shared_vars = sorted(part1_4["shared_vars"])
print(f"Shared var_ids (from Part 1.4 cache): {len(shared_vars)}")


Shared var_ids (from Part 1.4 cache): 5674


## Assumption check: is `description` constant per `var_id`?

Both dictionaries store one row per `(var_id, value_id)` pair (a row per
category code), so `description` must be verified constant within a
`var_id` before treating "the description for this var_id" as a
well-defined single string to compare.


In [2]:
desc24_nunique = vm2024.groupby("var_id")["description"].nunique()
desc26_nunique = vm2026.groupby("var_id")["description"].nunique()
print(f"2024H2: var_ids with >1 distinct description across their rows: {(desc24_nunique > 1).sum()} / {len(desc24_nunique)}")
print(f"2026:   var_ids with >1 distinct description across their rows: {(desc26_nunique > 1).sum()} / {len(desc26_nunique)}")
print("\n[OK] description is constant per var_id in both dictionaries -- a direct per-var_id text comparison is valid.")


2024H2: var_ids with >1 distinct description across their rows: 0 / 11878
2026:   var_ids with >1 distinct description across their rows: 0 / 10943

[OK] description is constant per var_id in both dictionaries -- a direct per-var_id text comparison is valid.


## Methodology

For each shared `var_id`, compare its 2024H2 description string to its 2026
description string:

- **`raw_exact`** — byte-identical strings.
- **`norm_exact`** — identical after normalizing whitespace/case/punctuation
  (catches purely cosmetic formatting differences, e.g. trailing whitespace
  or punctuation-only edits, that `raw_exact` would over-flag as a change).
- **`sim_ratio`** — `difflib.SequenceMatcher` ratio (0-1) on the normalized
  strings, as a continuous measure of how much wording changed for
  non-exact matches.

Each `var_id` is bucketed into one status:
- **IDENTICAL** — `raw_exact`
- **COSMETIC_ONLY** — `norm_exact` but not `raw_exact`
- **REWORDED** — `sim_ratio >= 0.7` (substantial word overlap; a paraphrase
  or minor rewrite of the same underlying question)
- **CHANGED** — `sim_ratio < 0.7` (little to no textual overlap; a
  different question in substance, not just phrasing)

The 0.7 cutoff is a convention chosen for this notebook (not a
domain-specific threshold derived from validation), analogous to Part
1.3's Cramér's V > 0.1 flagging convention — it separates "same question,
different words" from "different question" by eye on inspection of
examples near the boundary, not by a formal derivation.

Each `var_id` is also tagged with its 2024H2 `table_id`'s Part 1.1
`table_class`: **STABLE** (table-level description-set Jaccard = 1.000),
**UNSTABLE** (Jaccard = 0.0), or **OTHER** (partial overlap, 67 tables not
classified at either extreme) — to check whether variable-level drift is
confined to tables Part 1.1 already flagged, or also hides inside tables
Part 1.1 called stable.


In [3]:
def norm(s):
    s = str(s).strip().lower()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^\w\s]", "", s)
    return s

desc24 = vm2024.drop_duplicates("var_id").set_index("var_id")["description"]
desc26 = vm2026.drop_duplicates("var_id").set_index("var_id")["description"]
table24 = vm2024.drop_duplicates("var_id").set_index("var_id")["table_id"]

sim = pd.read_csv("data/part1_1_table_self_similarity.csv")
unstable_tables = set(sim.loc[sim["self_jaccard"] == 0.0, "table_id"])
stable_tables = set(sim.loc[sim["self_jaccard"] == 1.0, "table_id"])
print(f"Part 1.1 table classes -- STABLE: {len(stable_tables)}, UNSTABLE: {len(unstable_tables)}, "
      f"OTHER (partial overlap): {len(sim) - len(stable_tables) - len(unstable_tables)}")

rows = []
for vid in shared_vars:
    a, b = desc24.get(vid), desc26.get(vid)
    if pd.isna(a) or pd.isna(b):
        continue
    raw_exact = (a == b)
    norm_exact = (norm(a) == norm(b))
    ratio = difflib.SequenceMatcher(None, norm(a), norm(b)).ratio()
    if raw_exact:
        status = "IDENTICAL"
    elif norm_exact:
        status = "COSMETIC_ONLY"
    elif ratio >= 0.7:
        status = "REWORDED"
    else:
        status = "CHANGED"
    tbl = table24.get(vid)
    rows.append({
        "var_id": vid, "table_id": tbl, "desc_2024": a, "desc_2026": b,
        "raw_exact": raw_exact, "norm_exact": norm_exact, "sim_ratio": round(ratio, 3),
        "status": status,
        "table_class": "UNSTABLE" if tbl in unstable_tables else "STABLE" if tbl in stable_tables else "OTHER",
    })

res = pd.DataFrame(rows)
res.to_csv("data/part1_var_description_diff.csv", index=False)
print(f"\nCompared {len(res)} of {len(shared_vars)} shared var_ids (all 5,674 have a description in both dictionaries).")


Part 1.1 table classes -- STABLE: 135, UNSTABLE: 130, OTHER (partial overlap): 67



Compared 5674 of 5674 shared var_ids (all 5,674 have a description in both dictionaries).


## Observed results — headline numbers


In [4]:
counts = res["status"].value_counts()
pcts = res["status"].value_counts(normalize=True) * 100
summary = pd.DataFrame({"n": counts, "pct": pcts.round(1)})
print(summary)
print()
print("Cross-tab: status x Part 1.1 table_class")
print(pd.crosstab(res["status"], res["table_class"]))


                  n   pct
status                   
CHANGED        3380  59.6
REWORDED       1284  22.6
IDENTICAL      1009  17.8
COSMETIC_ONLY     1   0.0

Cross-tab: status x Part 1.1 table_class
table_class    OTHER  STABLE  UNSTABLE
status                                
CHANGED         1193     487      1700
COSMETIC_ONLY      0       0         1
IDENTICAL         69     940         0
REWORDED         659     460       165


**Observed:** of the 5,674 shared `var_id`s, only 1,009 (17.8%) have a
byte-identical description in both dictionaries; 1 (0.02%) differs by
formatting only. 1,284 (22.6%) are reworded but recognizably the same
question (`sim_ratio` 0.7-1.0), and **3,380 (59.6%) have descriptions with
little to no textual overlap** — the same `var_id` string names a
substantively different question in 2026 than it did in 2024H2.

As expected, essentially all of that churn concentrates in tables Part 1.1
already flagged (`UNSTABLE`, table-level Jaccard = 0.0): 1,700 of 1,865
shared vars in those tables are `CHANGED`, and **zero** are `IDENTICAL` —
fully consistent with, and a variable-level confirmation of, Part 1.1's
table-level finding.

## The new finding: reshuffled `var_id`s inside tables Part 1.1 called "stable"

The `STABLE` column is the important one: 947 of 1,887 shared vars (50.2%)
in tables Part 1.1 confirmed at **table-level description-set Jaccard =
1.000** are *not* `IDENTICAL` at the variable level (487 `CHANGED`, 460
`REWORDED`). That looks like a contradiction — if a table's overall
description Jaccard is 1.000, how can an individual variable's own
description differ? Investigated below.


In [5]:
quiet = res[(res["table_class"] == "STABLE") & (res["status"] != "IDENTICAL")].sort_values("sim_ratio")
print(f"Non-identical shared vars inside STABLE tables: {len(quiet)}")
print()
print(quiet[["var_id", "table_id", "desc_2024", "desc_2026", "sim_ratio", "status"]].head(12).to_string(index=False))


Non-identical shared vars inside STABLE tables: 947

   var_id table_id                                   desc_2024                                                          desc_2026  sim_ratio  status
 vv_mov_6   vv_mov           Went Last Time - In Past 2 Months        Type(s) Of Movies Attended - Any - Family/Children Oriented      0.093 CHANGED
 vv_mov_2   vv_mov               Went Last Time - In Past Week                Type(s) Of Movies Attended - Any - Action/Adventure      0.108 CHANGED
 vv_but_7   vv_but      Finance/Investment:  Employee Benefits                                      Services:  Education/Training      0.131 CHANGED
 vv_mov_7   vv_mov           Went Last Time - In Past 3 Months                         Type(s) Of Movies Attended - Any - Foreign      0.171 CHANGED
 vv_mov_9   vv_mov          Went Last Time - In Past 12 Months                 Type(s) Of Movies Attended - Any - Science Fiction      0.177 CHANGED
 vv_ups_1   vv_ups                             Used/P

In [6]:
# Verify: is this a reshuffle (content preserved elsewhere in the same table_id,
# under a different var_id) rather than content genuinely vanishing?
tbl_desc26 = vm2026.groupby("table_id")["description"].apply(set).to_dict()
tbl_desc24 = vm2024.groupby("table_id")["description"].apply(set).to_dict()

check = res[(res["table_class"] == "STABLE") & (res["status"] != "IDENTICAL")].copy()
check["desc24_found_elsewhere_in_2026_table"] = check.apply(
    lambda r: r["desc_2024"] in tbl_desc26.get(r["table_id"], set()), axis=1)
check["desc26_found_elsewhere_in_2024_table"] = check.apply(
    lambda r: r["desc_2026"] in tbl_desc24.get(r["table_id"], set()), axis=1)

print(f"Of {len(check)} non-identical shared vars in STABLE tables:")
print(f"  2024H2 description also appears somewhere in the SAME 2026 table (different var_id): "
      f"{check['desc24_found_elsewhere_in_2026_table'].sum()} ({check['desc24_found_elsewhere_in_2026_table'].mean():.1%})")
print(f"  2026 description also appears somewhere in the SAME 2024H2 table (different var_id): "
      f"{check['desc26_found_elsewhere_in_2024_table'].sum()} ({check['desc26_found_elsewhere_in_2024_table'].mean():.1%})")


Of 947 non-identical shared vars in STABLE tables:
  2024H2 description also appears somewhere in the SAME 2026 table (different var_id): 947 (100.0%)
  2026 description also appears somewhere in the SAME 2024H2 table (different var_id): 947 (100.0%)


**Confirmed: this is a 100% reshuffle, not vanished content.** Every one of
the 947 non-identical `STABLE`-table variables has its description text
preserved somewhere else in the *same* table (same `table_id`) in the other
vintage, just under a different `var_id` number. Concrete example
(`vv_mov`, 33/33 variables all reshuffled): `vv_mov_6` = "Went Last Time -
In Past 2 Months" in 2024H2, but = "Type(s) Of Movies Attended - Any -
Family/Children Oriented" in 2026 — a completely different question, even
though the set of 33 descriptions used across `vv_mov` is identical between
vintages (hence Part 1.1's 1.000 self-similarity score) and even though
`vv_mov_6` "exists" under the same name in both.

This is a distinct failure mode from anything Part 1.1 or Part 1.3 caught:

- **Part 1.1** (table-level, bag-of-descriptions Jaccard) is blind to this
  by construction — it only asks whether the *set* of content in a table
  changed, not whether that content stayed attached to the same `var_id`
  numbers.
- **Part 1.3** (variable-level *response-scale*/category-structure
  stability) checks a different property — whether a variable's *category
  set* changed — and would not catch a case where the category structure
  happens to match by coincidence (e.g. two Yes/No items swapping places)
  even though the *question* changed.

**How many tables does this affect?**


In [7]:
per_table = res[res["table_class"] == "STABLE"].groupby("table_id").apply(
    lambda g: pd.Series({
        "n_vars_checked": len(g),
        "n_identical": (g["status"] == "IDENTICAL").sum(),
        "n_reshuffled": (g["status"] != "IDENTICAL").sum(),
    }), include_groups=False,
)
n_checked = len(per_table)
n_fully_aligned = (per_table["n_reshuffled"] == 0).sum()
n_any_reshuffle = (per_table["n_reshuffled"] > 0).sum()
print(f"STABLE tables with >=1 shared var checked: {n_checked} (of 135 total STABLE tables)")
print(f"  fully aligned -- every var_id maps to the same description in both vintages: {n_fully_aligned}")
print(f"  with >=1 reshuffled var_id despite table-level jaccard=1.000: {n_any_reshuffle} "
      f"({n_any_reshuffle / n_checked:.1%})")
print()
print("Worst-affected STABLE tables (most reshuffled variables):")
print(per_table.sort_values("n_reshuffled", ascending=False).head(10))


STABLE tables with >=1 shared var checked: 135 (of 135 total STABLE tables)
  fully aligned -- every var_id maps to the same description in both vintages: 79
  with >=1 reshuffled var_id despite table-level jaccard=1.000: 56 (41.5%)

Worst-affected STABLE tables (most reshuffled variables):
          n_vars_checked  n_identical  n_reshuffled
table_id                                           
vv_nau                97           21            76
vv_win                77            6            71
vv_fur                62            0            62
vv_tve                60            0            60
vv_gar                47            7            40
vv_res                39            0            39
vv_mov                33            0            33
vv_vid                32            0            32
vv_yog                33            2            31
vv_ric                33            2            31


**56 of 135 (41.5%) tables Part 1.1 confirmed as fully content-stable have
at least one internally reshuffled `var_id`**, and in the worst cases
(`vv_fur`, `vv_tve`, `vv_res`, `vv_mov`, `vv_vid`) *every single shared
variable* in the table is reshuffled — the table's total content is
unchanged, but not one `var_id` number still points at the same question.
`vv_res` is a table this project has already flagged independently: Part
1.3 found all 39/39 of its variables fail the response-scale
(`categories_match`) check; this notebook now shows those same 39
variables are *also* fully reshuffled at the description level — `vv_res`
is broken on both axes simultaneously despite passing Part 1.1's
table-level content check.

As a sanity check in the other direction: `vv_dem` — the table Part 1.2's
ID-continuity check and Task 1's demographic sanity checks depend on being
stable — has all 20 of its shared variables `IDENTICAL` (verified directly,
not assumed), consistent with every other check in this project that
relies on `vv_dem`.

## Examples of the largest description changes overall (any table class)


In [8]:
worst = res.sort_values("sim_ratio").head(15)
print(worst[["var_id", "table_id", "desc_2024", "desc_2026", "sim_ratio", "table_class"]].to_string(index=False))


    var_id table_id                                                                                                                                              desc_2024                                                                         desc_2026  sim_ratio table_class
  vv_pro_5   vv_pro                                                                           When I buy any product, its style and design are as important as its quality                                                   Type(s) Hhld. Used - Most Often      0.059    UNSTABLE
  vv_auw_9   vv_auw                                                                                                             Given choice I'd always choose luxury auto Automotive Services Personally Had Done Past 12 Months - Tires repaired/installed      0.067    UNSTABLE
  vv_hea_2   vv_hea                                  Practitioners Visited in Past 6 Months/ Intend to Visit in Next 12 Months - Visited in Past 6 Months - Aromatherapist  

## Methodological assumptions

- **0.7 similarity cutoff (REWORDED vs. CHANGED)** is a convention for this
  notebook, chosen by inspecting examples near the boundary, not derived
  from a labeled ground truth of "same question" vs. "different question."
  A different cutoff would move variables between these two buckets, but
  would not change the headline finding (the reshuffle phenomenon is
  demonstrated by exact string membership, not by the similarity score).
- **Reshuffle verification uses set membership within a `table_id`**, not
  multiplicity. If a table has a duplicate description used by more than
  one `var_id` in one vintage, this check would still call it a "reshuffle"
  even if the true correspondence is ambiguous; not separately checked, and
  none of the tables inspected by name above showed within-vintage
  duplicate descriptions.
- Every shared `var_id` in `data/part1_4_diff_results.pkl`'s `shared_vars`
  set had a description available in both dictionaries (5,674 of 5,674) —
  no missing-description cases to separately handle.

## Conclusion

Part 1.1's table-level content check and Part 1.3's variable-level
response-scale check are each necessary but **neither is sufficient** to
certify that a specific `var_id` means the same thing across the 2024H2 to
2026 vintage change. This notebook adds a third, independent failure mode:
**even a `var_id` that "exists" under the same name in a table Part 1.1
confirmed fully content-stable can point at a different question
entirely** — 41.5% of stable tables have at least one such reshuffled
variable, and in the worst-affected tables every variable is reshuffled.
Across the full 5,674-variable shared set, only 17.8% have a byte-identical
description in both vintages; 59.6% differ enough in wording to represent a
different question in substance.

**Practical implication (extends Part 1.4's conclusion):** any process
that references a 2024-vintage `var_id` against 2026 data — this project's
own fusion pipeline included — cannot treat "the var_id string is
unchanged" as evidence of anything, even when the *table* has already been
confirmed content-stable by Part 1.1. The only verification that holds up
is checking each specific `var_id`'s description text directly, which is
what this notebook did exhaustively for the full shared set rather than a
sample.
